# Phase 3 Part 3: SimGCL on All 5 Datasets
**CSC14114 - Big Data Applications | Official QRec (Coder-Yu/QRec) SimGCL workflow**

Uses the official QRec repository with TF2 compat patches for Kaggle.

## Kaggle setup
- Internet ON, GPU ON
- Attach Last.fm Phase 2 dataset
- Run All → Save Version


In [ ]:
import json, os, random, re, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, requests

ROOT = Path('/kaggle/working/phase3_part3_simgcl') if Path('/kaggle').exists() else Path('phase3_part3_simgcl')
WORK = ROOT / 'workspace'
REPO_DIR = WORK / 'QRec'
RESULTS_DIR = ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
INPUT_DIR = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
for p in [WORK, RESULTS_DIR, FIG_DIR]: p.mkdir(parents=True, exist_ok=True)
print('Workspace:', ROOT.resolve())


In [ ]:
LASTFM_REQUIRED = ['interactions.csv','users.csv','items.csv','manifest.json']

def find_lastfm(input_dir):
    for zp in sorted(input_dir.rglob('*.zip')):
        try:
            with zipfile.ZipFile(zp) as zf:
                names = {Path(n).name for n in zf.namelist() if not n.endswith('/')}
            if all(n in names for n in LASTFM_REQUIRED): return ('zip', zp)
        except zipfile.BadZipFile: continue
    for mp in sorted(input_dir.rglob('manifest.json')):
        f = mp.parent
        if all((f/n).exists() for n in LASTFM_REQUIRED): return ('dir', f)
    raise FileNotFoundError('Last.fm Phase 2 data not found')

def materialize_lastfm():
    t = WORK / 'lastfm_phase2_input'
    if t.exists() and all((t/n).exists() for n in LASTFM_REQUIRED): return t
    if t.exists(): shutil.rmtree(t)
    t.mkdir(parents=True, exist_ok=True)
    kind, src = find_lastfm(INPUT_DIR)
    if kind == 'zip':
        with zipfile.ZipFile(src) as zf: zf.extractall(t)
    else:
        for n in LASTFM_REQUIRED: shutil.copy2(src/n, t/n)
    return t

LASTFM_DIR = materialize_lastfm()
print(f'Last.fm ready: {LASTFM_DIR}')


In [ ]:
REPO_URL = 'https://github.com/Coder-Yu/QRec.git'
SEEDS = [42, 42, 42]
TOPKS = [10, 20]
CFG = {
    'ml-1m':         {'epochs':200,'batch_size':2048,'emb_size':64,'n_layer':3,'lr':0.001,'reg':1e-4,'eps':0.1,'cl_lambda':0.5},
    'gowalla':       {'epochs':200,'batch_size':2048,'emb_size':64,'n_layer':3,'lr':0.001,'reg':1e-4,'eps':0.1,'cl_lambda':0.5},
    'yelp2018':      {'epochs':200,'batch_size':2048,'emb_size':64,'n_layer':2,'lr':0.001,'reg':1e-4,'eps':0.1,'cl_lambda':0.5},
    'amazon-book':   {'epochs':200,'batch_size':2048,'emb_size':64,'n_layer':2,'lr':0.001,'reg':1e-4,'eps':0.1,'cl_lambda':0.5},
    'lastfm_phase2': {'epochs':200,'batch_size':2048,'emb_size':64,'n_layer':3,'lr':0.001,'reg':1e-4,'eps':0.1,'cl_lambda':0.5},
}
RAW_URLS = {
    'gowalla':     {'train':'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/train.txt',
                    'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/test.txt'},
    'yelp2018':    {'train':'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/train.txt',
                    'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/test.txt'},
    'amazon-book': {'train':'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/train.txt',
                    'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/test.txt'},
}
ML_URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'


In [ ]:
def run_cmd(cmd, **kw):
    print('$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, check=True, text=True, **kw)

def download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists(): return dest
    print(f'[dl] {url}')
    r = requests.get(url, stream=True, timeout=300); r.raise_for_status()
    with dest.open('wb') as f:
        for c in r.iter_content(1024*1024):
            if c: f.write(c)
    return dest

def lgcn_to_qrec(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    with src.open() as fin, dst.open('w') as fout:
        for line in fin:
            p = line.strip().split()
            if len(p) < 2: continue
            for iid in p[1:]: fout.write(f'{p[0]} {iid} 1\n')

def loo_qrec(df, uc, ic, tc, folder):
    folder.mkdir(parents=True, exist_ok=True)
    w = df[[uc,ic,tc]].copy().sort_values([uc,tc,ic], kind='mergesort')
    um = {u:i for i,u in enumerate(sorted(w[uc].unique()))}
    im = {t:i for i,t in enumerate(sorted(w[ic].unique()))}
    w['uid'] = w[uc].map(um).astype(int)
    w['iid'] = w[ic].map(im).astype(int)
    with (folder/'train.txt').open('w') as ft, (folder/'test.txt').open('w') as fe:
        for uid, g in w.groupby('uid', sort=True):
            items = g['iid'].tolist()
            if len(items) < 2: continue
            for iid in items[:-1]: ft.write(f'{uid} {iid} 1\n')
            fe.write(f'{uid} {items[-1]} 1\n')


In [ ]:
def clone_and_patch():
    marker = REPO_DIR / '.patched_v3'
    if REPO_DIR.exists() and not marker.exists():
        shutil.rmtree(REPO_DIR)
    if not REPO_DIR.exists():
        run_cmd(['git','clone', REPO_URL, str(REPO_DIR)])
    if marker.exists():
        print('QRec already patched.'); return

    # Patch TF imports
    for rel in ['model/ranking/SimGCL.py','base/graphRecommender.py','base/deepRecommender.py',
                'base/iterativeRecommender.py','base/recommender.py','util/loss.py']:
        fp = REPO_DIR / rel
        if not fp.exists(): continue
        t = fp.read_text('utf-8')
        t = t.replace('import tensorflow as tf',
                      'import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()')
        t = t.replace('tf.contrib.layers.xavier_initializer()', 'tf.initializers.glorot_uniform()')
        fp.write_text(t, 'utf-8')

    # Patch QRec.py: remove mkl
    qp = REPO_DIR / 'QRec.py'
    t = qp.read_text('utf-8')
    t = t.replace('import mkl\n', '# import mkl (patched)\nmkl = None\n')
    t = t.replace('mkl.set_num_threads(max(1,mkl.get_max_threads()//k))', 'pass  # mkl removed')
    qp.write_text(t, 'utf-8')

    marker.write_text('ok')
    print('Patched QRec for TF2.')

clone_and_patch()
run_cmd([sys.executable,'-m','pip','install','-q','scipy','numba'])
print('QRec ready.')


In [ ]:
def prep_standard():
    tmp = WORK / 'lgcn_raw'
    for name, urls in RAW_URLS.items():
        qf = REPO_DIR / 'dataset' / name
        raw = tmp / name; raw.mkdir(parents=True, exist_ok=True)
        download(urls['train'], raw/'train.txt')
        download(urls['test'], raw/'test.txt')
        qf.mkdir(parents=True, exist_ok=True)
        lgcn_to_qrec(raw/'train.txt', qf/'train.txt')
        lgcn_to_qrec(raw/'test.txt', qf/'test.txt')
        print(f'OK {name}')

def prep_ml1m():
    f = REPO_DIR/'dataset'/'ml-1m'
    if (f/'train.txt').exists(): print('ml-1m exists'); return
    zp = download(ML_URL, WORK/'ml-1m.zip')
    ext = WORK/'ml-1m_ext'
    if not ext.exists():
        with zipfile.ZipFile(zp) as zf: zf.extractall(ext)
    df = pd.read_csv(ext/'ml-1m'/'ratings.dat', sep='::', header=None,
                     names=['user_id','item_id','rating','timestamp'], engine='python', encoding='latin-1')
    df = df[['user_id','item_id','timestamp']].drop_duplicates(['user_id','item_id'])
    loo_qrec(df,'user_id','item_id','timestamp',f)
    print('OK ml-1m')

def prep_lastfm():
    f = REPO_DIR/'dataset'/'lastfm_phase2'
    if (f/'train.txt').exists(): print('lastfm exists'); return
    df = pd.read_csv(LASTFM_DIR/'interactions.csv')
    df['timestamp'] = pd.to_datetime(df['crawl_timestamp_utc'], utc=True, errors='coerce')
    df = df.dropna(subset=['timestamp']).copy()
    df['timestamp'] = df['timestamp'].astype('int64')//10**9
    df = df[['user_id','item_id','timestamp']].drop_duplicates(['user_id','item_id'])
    loo_qrec(df,'user_id','item_id','timestamp',f)
    print('OK lastfm_phase2')

prep_standard()
prep_ml1m()
prep_lastfm()
print('Datasets:')
for p in sorted((REPO_DIR/'dataset').iterdir()):
    if p.is_dir(): print('-', p.name)


In [ ]:
def write_conf(dataset, seed, run_dir):
    c = CFG[dataset]
    dd = str((REPO_DIR/'dataset'/dataset).resolve())
    od = str((run_dir/'qrec_output').resolve())
    Path(od).mkdir(parents=True, exist_ok=True)
    topk = ','.join(str(k) for k in TOPKS)
    # NOTE: -columns 0 1 2 (all 3 cols present in our data: uid iid 1)
    # NOTE: NO -b flag (data is already binary, avoids QRec binarize bug)
    txt = (
        f'ratings={dd}/train.txt\n'
        f'ratings.setup=-columns 0 1 2\n'
        f'model.name=SimGCL\n'
        f'evaluation.setup=-testSet {dd}/test.txt\n'
        f'item.ranking=on -topN {topk}\n'
        f'num.factors={c["emb_size"]}\n'
        f'num.max.epoch={c["epochs"]}\n'
        f'batch_size={c["batch_size"]}\n'
        f'learnRate=-init {c["lr"]} -max 1\n'
        f'reg.lambda=-u {c["reg"]} -i {c["reg"]} -b 0.2 -s 0.2\n'
        f'SimGCL=-n_layer {c["n_layer"]} -lambda {c["cl_lambda"]} -eps {c["eps"]}\n'
        f'output.setup=on -dir {od}/\n'
    )
    path = run_dir / 'SimGCL.conf'
    path.write_text(txt, encoding='utf-8')
    return path


In [ ]:
def parse_log(log):
    rows = []
    for line in log.splitlines():
        if 'Epoch:' not in line or 'Current' in line or 'Best' in line: continue
        em = re.search(r'Epoch:\s*(\d+)', line)
        if not em: continue
        row = {'eval_epoch': int(em.group(1))}
        for m in re.finditer(r'(Hit Ratio|Precision|Recall|F1|NDCG)\s*:\s*([\d.]+)', line, re.I):
            row[m.group(1).lower().replace(' ','_')] = float(m.group(2))
        if len(row) > 1: rows.append(row)
    return rows

def run_one(dataset, seed):
    rd = RESULTS_DIR/dataset/f'seed_{seed}'
    rd.mkdir(parents=True, exist_ok=True)
    conf = write_conf(dataset, seed, rd)
    repo = str(REPO_DIR.resolve())
    cstr = str(conf.resolve())

    code = (
        'import sys, os\n'
        f'sys.path.insert(0, "{repo}")\n'
        f'os.chdir("{repo}")\n'
        'import random, numpy as np\n'
        f'random.seed({seed}); np.random.seed({seed})\n'
        'import tensorflow.compat.v1 as tf\n'
        'tf.disable_v2_behavior()\n'
        f'tf.set_random_seed({seed})\n'
        'from util.config import ModelConf\n'
        'from QRec import QRec\n'
        f'conf = ModelConf("{cstr}")\n'
        'rec = QRec(conf)\n'
        'rec.execute()\n'
    )
    script = rd / '_run.py'
    script.write_text(code, encoding='utf-8')

    print('='*90)
    print(f'SimGCL (QRec): {dataset}, seed={seed}')
    print('='*90)
    t0 = time.time()
    proc = subprocess.run([sys.executable, str(script)], text=True, capture_output=True)
    elapsed = round(time.time()-t0, 2)
    log = (proc.stdout or '')+'\n'+(proc.stderr or '')
    (rd/'train.log').write_text(log, encoding='utf-8')
    if proc.returncode != 0:
        print('--- STDOUT ---'); print((proc.stdout or '')[-3000:])
        print('--- STDERR ---'); print((proc.stderr or '')[-3000:])
        raise RuntimeError(f'Failed: {dataset}/{seed}')

    h = parse_log(log)
    pd.DataFrame(h).to_csv(rd/'history.csv', index=False)
    final = h[-1] if h else {}
    result = {'dataset':dataset,'seed':seed,'elapsed_seconds':elapsed, **final}
    (rd/'metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    return result


In [ ]:
all_results = []
for ds in ['ml-1m','gowalla','yelp2018','amazon-book','lastfm_phase2']:
    for seed in SEEDS:
        all_results.append(run_one(ds, seed))
results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_DIR/'all_runs.csv', index=False)
display(results_df)


In [ ]:
mc = [c for c in results_df.columns if c in ['recall','ndcg','precision','hit_ratio','f1']]+['elapsed_seconds']
rows = []
for ds,g in results_df.groupby('dataset'):
    r = {'dataset':ds,'runs':len(g)}
    for c in mc:
        if c in g.columns: r[f'{c}_mean']=float(g[c].mean()); r[f'{c}_std']=float(g[c].std(ddof=0))
    rows.append(r)
summary_df = pd.DataFrame(rows).sort_values('dataset').reset_index(drop=True)
summary_df.to_csv(RESULTS_DIR/'summary.csv', index=False)
display(summary_df)


In [ ]:
def stats(ds):
    f = REPO_DIR/'dataset'/ds
    def p(path):
        u,i,n = set(),set(),0
        for line in path.read_text('utf-8').splitlines():
            pp = line.strip().split()
            if len(pp)<2: continue
            u.add(pp[0]); i.add(pp[1]); n+=1
        return u,i,n
    tu,ti,tn = p(f/'train.txt'); eu,ei,en = p(f/'test.txt')
    au,ai = tu|eu, ti|ei; tot = tn+en
    return {'dataset':ds,'users':len(au),'items':len(ai),'train':tn,'test':en,'total':tot,
            'density':tot/(len(au)*len(ai)) if au and ai else 0}

sdf = pd.DataFrame([stats(d) for d in CFG])
sdf.to_csv(RESULTS_DIR/'dataset_characteristics.csv', index=False)
adf = summary_df.merge(sdf, on='dataset', how='left')
adf.to_csv(RESULTS_DIR/'analysis.csv', index=False)
display(sdf)

fig, axes = plt.subplots(2,2,figsize=(14,10))
for ax,(col,title,color) in zip(axes.flatten(),[
    ('recall_mean','Recall','#3366cc'),('ndcg_mean','NDCG','#dc3912'),
    ('elapsed_seconds_mean','Runtime(s)','#ff9900'),('density','Density','#109618')]):
    if col not in adf.columns: ax.set_title(f'{title} (n/a)'); continue
    bars = ax.bar(adf['dataset'], adf[col], color=color)
    ax.set_title(title, fontweight='bold'); ax.tick_params(axis='x', rotation=20)
    for b,v in zip(bars,adf[col]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height(), f'{v:.4f}' if v<1000 else f'{v:,.0f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR/'overview.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
for hp in sorted(RESULTS_DIR.glob('*/*/history.csv')):
    hdf = pd.read_csv(hp)
    if hdf.empty: continue
    ds, sd = hp.parent.parent.name, hp.parent.name
    plt.figure(figsize=(7,4))
    if 'recall' in hdf.columns: plt.plot(hdf['eval_epoch'], hdf['recall'], label='Recall')
    if 'ndcg' in hdf.columns: plt.plot(hdf['eval_epoch'], hdf['ndcg'], label='NDCG')
    plt.xlabel('Epoch'); plt.title(f'SimGCL | {ds} - {sd}'); plt.legend()
    plt.tight_layout(); plt.savefig(FIG_DIR/f'{ds}_{sd}.png', dpi=150, bbox_inches='tight'); plt.close()

meta = {'model':'SimGCL','repo':REPO_URL,'datasets':list(CFG),'seeds':SEEDS,'topks':TOPKS,
        'configs':{k:{kk:str(vv) for kk,vv in v.items()} for k,v in CFG.items()}}
(RESULTS_DIR/'metadata.json').write_text(json.dumps(meta, indent=2))
ar = ROOT/'phase3_part3_results'
if ar.with_suffix('.zip').exists(): ar.with_suffix('.zip').unlink()
shutil.make_archive(str(ar), 'zip', RESULTS_DIR)
print(f'Archive: {ar.with_suffix(".zip")}')
print('Done.')
